# NumPy: From Beginner to Highly Proficient
### A practical, exercise-driven reference and learning course

This notebook is designed to take you from the first NumPy array to writing efficient, vectorized numerical code used in data science and machine learning.

**How to use it**
1. Work through the sections in order. Run every code cell yourself.
2. Before running an example, predict its output.
3. Complete the exercises without immediately looking at the solutions.
4. Change array shapes and values to see how the rules behave.
5. For mastery, re-implement small examples from memory and explain *why* they work.

**Important:** NumPy is a large library. This notebook covers the core API and the most useful surrounding tools, not literally every obscure function. Use `help(np.some_function)` and the official documentation when you encounter a specialized need.

**Suggested pace:** 3–6 weeks alongside school, depending on how much time you can consistently give it. Understanding and practice matter more than rushing.

## 0. Setup and the NumPy mental model

NumPy provides the `ndarray`: a multidimensional, usually homogeneous array. Arrays store values efficiently and let you perform operations on whole collections of values without writing Python loops.

The most important concepts to understand are:
- **shape:** the size along each dimension
- **ndim:** number of dimensions
- **size:** total number of elements
- **dtype:** the data type of the elements
- **axis:** the dimension along which an operation is performed
- **vectorization:** expressing work as array operations
- **broadcasting:** rules that let compatible shapes work together
- **views vs copies:** whether an operation shares the original data or creates new data

In [ ]:
import numpy as np

print("NumPy version:", np.__version__)

## 1. Creating arrays

These are the constructors you will use constantly.

In [ ]:
# From Python sequences
a = np.array([1, 2, 3, 4])
matrix = np.array([[1, 2, 3], [4, 5, 6]])

# Common filled arrays
zeros = np.zeros((2, 3))
ones = np.ones((2, 3))
full = np.full((2, 3), 7)
identity = np.eye(3)

# Sequences
range_array = np.arange(0, 10, 2)       # stop is excluded
evenly_spaced = np.linspace(0, 1, 5)    # 5 values including endpoints

# Uninitialized allocation: values are arbitrary; fill before reading.
empty = np.empty((2, 2))

print(a)
print(matrix)
print(range_array)
print(evenly_spaced)

### More constructors and conversion tools

- `np.asarray(x)`: convert to an array, avoiding a copy when possible.
- `np.asanyarray(x)`: like `asarray`, but preserves ndarray subclasses.
- `np.copy(x)`: explicit copy.
- `np.zeros_like(x)`, `ones_like(x)`, `full_like(x)`: create arrays matching another array's shape.
- `np.fromiter(iterable, dtype=...)`: build an array from an iterable.
- `np.fromfunction(func, shape)`: create values using coordinate indices.
- `np.meshgrid(x, y)`: create coordinate grids.
- `np.diag(v)`: construct or extract a diagonal.
- `np.triu(a)` / `np.tril(a)`: upper/lower triangular parts.

In [ ]:
x = np.array([[1, 2], [3, 4]])
print(np.zeros_like(x))
print(np.full_like(x, 9))
print(np.diag([1, 2, 3]))
print(np.triu(np.arange(1, 10).reshape(3, 3)))

## 2. Understanding arrays: attributes, indexing and slicing

In [ ]:
x = np.arange(24).reshape(2, 3, 4)

print("shape:", x.shape)
print("dimensions:", x.ndim)
print("number of elements:", x.size)
print("data type:", x.dtype)
print("bytes per element:", x.itemsize)
print("total bytes:", x.nbytes)

# Indexing
v = np.array([10, 20, 30, 40, 50])
print(v[0], v[-1])

# Slicing: start:stop:step
print(v[1:4])
print(v[::2])
print(v[::-1])

# Multidimensional indexing
m = np.arange(12).reshape(3, 4)
print(m[1, 2])       # row 1, column 2
print(m[0, :])       # first row
print(m[:, 1])       # second column
print(m[1:, :2])     # submatrix

### Integer-array and Boolean indexing

Integer-array indexing selects specific positions. Boolean indexing selects entries where a condition is true. These are fundamental for filtering datasets.

In [ ]:
x = np.array([3, 8, 1, 9, 4, 7])
indices = np.array([0, 3, 5])
print(x[indices])

mask = x >= 5
print(mask)
print(x[mask])
print(x[x >= 5])

# Combine conditions with &, |, and ~. Parentheses are essential.
print(x[(x >= 4) & (x < 9)])

**Indexing cautions**
- Basic slices usually produce **views**; modifying the slice can modify the original.
- Integer-array and Boolean indexing produce **copies**.
- Use `a.copy()` when you need independent data.
- Use `np.shares_memory(a, b)` to test whether two arrays share memory.

In [ ]:
a = np.arange(6)
view = a[1:4]
view[0] = 99
print("Original changed:", a)

b = np.arange(6)
selected = b[[1, 2, 3]]
selected[0] = 99
print("Original unchanged:", b)

print(np.shares_memory(a, view))

## 3. Reshaping, combining and rearranging

Core functions:
- `reshape`, `ravel`, `flatten`
- `transpose`, `.T`, `swapaxes`, `moveaxis`
- `squeeze`, `expand_dims`, `atleast_1d/2d/3d`
- `concatenate`, `stack`, `vstack`, `hstack`, `dstack`, `column_stack`
- `split`, `array_split`, `hsplit`, `vsplit`
- `repeat`, `tile`, `flip`, `fliplr`, `flipud`, `roll`

In [ ]:
a = np.arange(12)
matrix = a.reshape(3, 4)
print(matrix)

print(matrix.T)                       # transpose
print(matrix.reshape(2, 6))
print(matrix.ravel())                 # flattened view when possible
print(matrix.flatten())               # always returns a copy

single = np.array([[[5, 6, 7]]])
print(np.squeeze(single).shape)
print(np.expand_dims([1, 2, 3], axis=0).shape)

left = np.array([[1, 2], [3, 4]])
right = np.array([[5, 6], [7, 8]])
print(np.concatenate([left, right], axis=0))
print(np.concatenate([left, right], axis=1))
print(np.stack([left, right], axis=0).shape)
print(np.vstack([left, right]))
print(np.hstack([left, right]))

print(np.split(np.arange(12), 3))
print(np.repeat([1, 2], 3))
print(np.tile([1, 2], 3))

## 4. Data types and type conversion

Common dtypes include `int8`, `int16`, `int32`, `int64`, `uint8`, `float16`, `float32`, `float64`, and `bool_`.

Choose dtypes deliberately:
- Smaller dtypes use less memory but have narrower ranges or less precision.
- `float32` is common in machine learning; `float64` is NumPy's usual default for many floating-point operations.
- Integer overflow can occur with fixed-width integer types.
- Converting floats to integers truncates the fractional part; it does not round.

In [ ]:
x = np.array([1, 2, 3], dtype=np.int32)
y = x.astype(np.float32)
print(x.dtype, y.dtype)

floats = np.array([1.9, 2.8, -1.2])
print(floats.astype(np.int64))
print(np.round(floats))

print(np.iinfo(np.int16))
print(np.finfo(np.float32))

## 5. Arithmetic, universal functions and vectorization

NumPy's **ufuncs** apply elementwise operations efficiently. Common ones include:
- Arithmetic: `add`, `subtract`, `multiply`, `divide`, `floor_divide`, `power`, `mod`
- Roots/logs: `sqrt`, `square`, `exp`, `log`, `log10`, `log1p`
- Trigonometry: `sin`, `cos`, `tan`, `arcsin`, `arctan2`
- Rounding: `round`, `floor`, `ceil`, `trunc`
- Comparisons: `greater`, `less`, `equal`, `not_equal`
- Logical: `logical_and`, `logical_or`, `logical_not`
- Bitwise: `bitwise_and`, `bitwise_or`, `invert`

Array operators are generally elementwise, not matrix multiplication.

In [ ]:
a = np.array([1, 2, 3])
b = np.array([10, 20, 30])

print(a + b)
print(a * b)
print(a ** 2)
print(np.sqrt(b))
print(np.exp(a))
print(np.log(b))
print(np.sin(a))
print(a > 1)

# Vectorized formula: no explicit Python loop required.
temperatures_c = np.array([0, 10, 20, 30])
temperatures_f = temperatures_c * 9 / 5 + 32
print(temperatures_f)

### Aggregation and reductions

Important reductions:
- `sum`, `prod`, `min`, `max`
- `mean`, `median`, `std`, `var`
- `argmin`, `argmax`
- `any`, `all`
- `ptp` (range: max minus min)
- `count_nonzero`

The `axis` parameter is essential. For a 2D array, `axis=0` reduces down rows and leaves one result per column; `axis=1` reduces across columns and leaves one result per row.

In [ ]:
data = np.array([[1, 2, 3], [4, 5, 6]])

print("total:", np.sum(data))
print("column totals:", np.sum(data, axis=0))
print("row totals:", np.sum(data, axis=1))
print("column means:", np.mean(data, axis=0))
print("maximum location:", np.argmax(data))
print("maximum per row:", np.argmax(data, axis=1))
print("any > 5:", np.any(data > 5))
print("all positive:", np.all(data > 0))

### Keeping dimensions with `keepdims`

`keepdims=True` retains reduced dimensions, often making later broadcasting easier.

In [ ]:
x = np.array([[1., 2., 3.], [4., 5., 6.]])
column_means = x.mean(axis=0, keepdims=True)
centered = x - column_means
print(column_means.shape)
print(centered)

## 6. Broadcasting

Broadcasting lets NumPy combine arrays with different shapes without manually copying data.

Compare dimensions from **right to left**. Two dimensions are compatible when they are equal or one of them is 1. Missing leading dimensions are treated as 1.

If a dimension pair is incompatible, NumPy raises a `ValueError`.

In [ ]:
# (3, 1) and (1, 4) broadcast to (3, 4)
a = np.array([[1], [2], [3]])
b = np.array([[10, 20, 30, 40]])
print(a + b)

# Add a feature offset to every row
batch = np.array([[1., 2., 3.], [4., 5., 6.]])
offset = np.array([100., 200., 300.])
print(batch + offset)

# Normalize each column using broadcasting
means = batch.mean(axis=0, keepdims=True)
stds = batch.std(axis=0, keepdims=True)
normalized = (batch - means) / stds
print(normalized)

**Broadcasting debugging habit:** print `.shape` for every participating array. Use `np.newaxis` (same as `None`) to insert a length-one dimension:

In [ ]:
x = np.arange(3)       # (3,)
row = x[np.newaxis, :] # (1, 3)
col = x[:, np.newaxis] # (3, 1)
print((row + col).shape)
print(row + col)

## 7. Sorting, searching, selecting and counting

Useful functions:
- `sort`, `argsort`, `partition`, `argpartition`
- `searchsorted`
- `where`, `select`, `choose`
- `unique`, `isin`, `in1d` (legacy-style API; prefer `isin`)
- `nonzero`, `argwhere`
- `bincount`, `digitize`
- `take`, `take_along_axis`, `put`, `put_along_axis`
- `clip`

In [ ]:
x = np.array([8, 2, 5, 2, 9, 1])
print(np.sort(x))
print(np.argsort(x))             # indices that would sort x
print(np.partition(x, 2))        # smallest 3 values occupy first 3 slots, unordered
print(np.unique(x, return_counts=True))

print(np.where(x > 4, "large", "small"))
print(np.nonzero(x > 4))
print(np.argwhere(x > 4))
print(np.isin(x, [2, 9]))
print(np.clip(x, 3, 7))
print(np.bincount(np.array([0, 1, 1, 2, 2, 2])))

## 8. Missing values and numerical safety

NumPy commonly represents missing floating-point data with `np.nan`. It also has `np.inf` and `-np.inf`.

Use:
- `isnan`, `isfinite`, `isinf`
- `nanmean`, `nanmedian`, `nanstd`, `nansum`, `nanmin`, `nanmax`
- `nan_to_num`
- `errstate` to control floating-point warning behavior

`NaN` is not equal to itself. Use `np.isnan(x)`, not `x == np.nan`.

In [ ]:
x = np.array([1.0, np.nan, 3.0, np.inf])
print(np.isnan(x))
print(np.isfinite(x))
print(np.nanmean(x))
print(np.nan_to_num(x, nan=0.0, posinf=999.0))

with np.errstate(divide="ignore", invalid="ignore"):
    result = np.array([1.0, 0.0]) / np.array([0.0, 0.0])
print(result)

## 9. Random numbers and reproducibility

Use the modern random generator API: `np.random.default_rng(seed)`. A seed makes a sequence reproducible. Avoid relying on global random state for new code.

Useful generator methods:
- `random`, `integers`
- `uniform`, `normal`, `standard_normal`
- `choice`, `permutation`, `shuffle`
- `binomial`, `poisson`, `exponential`
- `multivariate_normal`

In [ ]:
rng = np.random.default_rng(42)

print(rng.random(4))
print(rng.integers(0, 10, size=5))
print(rng.normal(loc=0, scale=1, size=5))
print(rng.choice([10, 20, 30], size=4, replace=True))

values = np.arange(10)
print(rng.permutation(values))

# Random train/test split indices
indices = rng.permutation(100)
train_idx = indices[:80]
test_idx = indices[80:]
print(len(train_idx), len(test_idx))

## 10. Linear algebra

For machine learning and scientific computing, understand these thoroughly:
- `@` and `np.matmul`: matrix multiplication
- `np.dot`: dot product / matrix product with dimensionality-dependent behavior
- `np.linalg`: linear algebra routines
- `np.outer`, `np.inner`, `np.cross`, `np.vdot`
- `np.trace`, `np.diag`

Important `np.linalg` functions:
- `norm`, `det`, `matrix_rank`
- `inv`, `pinv`
- `solve`, `lstsq`
- `eig`, `eigh`, `eigvals`, `eigvalsh`
- `svd`, `qr`
- `cholesky`
- `matrix_power`
- `cond`

Prefer `np.linalg.solve(A, b)` to explicitly calculating `inv(A) @ b`.

In [ ]:
A = np.array([[3., 1.], [1., 2.]])
b = np.array([9., 8.])

print("A @ b:", A @ b)
print("solution:", np.linalg.solve(A, b))
print("determinant:", np.linalg.det(A))
print("inverse:", np.linalg.inv(A))
print("rank:", np.linalg.matrix_rank(A))
print("eigenvalues:", np.linalg.eigvals(A))
print("norm:", np.linalg.norm(b))

U, S, VT = np.linalg.svd(A)
print("singular values:", S)

### Linear algebra shapes

A vector with shape `(n,)` is not the same thing as a column matrix with shape `(n, 1)`. This distinction matters when using matrix multiplication and broadcasting.

In [ ]:
v = np.array([1., 2., 3.])        # (3,)
column = v[:, None]               # (3, 1)
row = v[None, :]                  # (1, 3)
print(v.shape, column.shape, row.shape)
print(column @ row)               # outer-product matrix
print(row @ column)               # 1x1 matrix
print(np.outer(v, v))

## 11. Statistics and descriptive analysis

Core statistical tools:
- `mean`, `average` (supports weights), `median`
- `std`, `var`
- `percentile`, `quantile`
- `corrcoef`, `cov`
- `histogram`, `histogram2d`, `histogramdd`
- `bincount`

Remember that NumPy's default standard deviation uses `ddof=0` (population standard deviation). For the common sample estimate, use `ddof=1`.

In [ ]:
rng = np.random.default_rng(7)
scores = rng.normal(70, 12, size=1000)

print("mean:", np.mean(scores))
print("median:", np.median(scores))
print("population std:", np.std(scores))
print("sample std:", np.std(scores, ddof=1))
print("90th percentile:", np.percentile(scores, 90))
print("weighted average:", np.average([80, 90, 70], weights=[1, 2, 1]))

hist, edges = np.histogram(scores, bins=10)
print(hist)
print(edges)

## 12. Working with dates and times

NumPy has `datetime64` and `timedelta64`. These are useful for compact date arrays and time differences. For timezone-aware and more complex calendar work, Python's `datetime` or pandas may be more appropriate.

In [ ]:
dates = np.array(["2026-01-01", "2026-01-05", "2026-01-10"], dtype="datetime64[D]")
print(dates)
print(dates[1] - dates[0])
print(dates + np.timedelta64(7, "D"))
print(np.is_busday(dates))

## 13. String operations

NumPy supports fixed-width string dtypes and vectorized string operations. For extensive text processing, Python strings or pandas are often more convenient.

Explore `np.char` functions such as `add`, `lower`, `upper`, `strip`, `replace`, `split`, `startswith`, and `find`.

In [ ]:
words = np.array([" NumPy ", "Python", "MACHINE LEARNING"])
print(np.char.strip(words))
print(np.char.lower(words))
print(np.char.replace(words, " ", "_"))
print(np.char.startswith(words, "P"))

## 14. Structured arrays and record-style data

Structured dtypes let each element contain named fields with potentially different types. They are useful for specialized binary data and low-level work, but ordinary arrays or pandas DataFrames are usually easier for tabular analysis.

In [ ]:
people_dtype = np.dtype([("name", "U12"), ("age", "i4"), ("score", "f4")])
people = np.array([("Asha", 14, 91.5), ("Dev", 15, 88.0)], dtype=people_dtype)
print(people["name"])
print(people["score"])

## 15. Performance: write NumPy-style code

A major reason to use NumPy is to move loops from Python into optimized array operations.

Good habits:
- Prefer vectorized operations over Python loops for numerical work.
- Avoid repeatedly growing arrays with `np.append`; collect values and concatenate once.
- Preallocate arrays when the output size is known.
- Choose sensible dtypes.
- Avoid unnecessary copies.
- Use broadcasting rather than manually repeating arrays.
- Measure performance with realistic inputs; do not assume a clever-looking expression is faster.

In [ ]:
# Compare a Python loop with a vectorized expression.
import time

rng = np.random.default_rng(0)
x = rng.random(1_000_000)

start = time.perf_counter()
loop_result = [value * value + 2 * value for value in x]
loop_time = time.perf_counter() - start

start = time.perf_counter()
vector_result = x * x + 2 * x
vector_time = time.perf_counter() - start

print(f"Python loop: {loop_time:.4f} seconds")
print(f"NumPy vectorized: {vector_time:.4f} seconds")
print("Results agree:", np.allclose(loop_result, vector_result))

### `out`, in-place operations and memory

Many ufuncs accept `out=` to write into an existing array. In-place operators such as `+=` may avoid allocating a new result, but they mutate the array.

In [ ]:
x = np.arange(5, dtype=float)
out = np.empty_like(x)
np.multiply(x, 10, out=out)
print(out)

x += 1
print(x)

## 16. Advanced indexing patterns

These patterns show up in data preprocessing and machine learning:
- Row/column selection with integer arrays
- Boolean filtering
- Reordering examples with a permutation
- Gathering values with `take_along_axis`
- Updating selected values
- Creating coordinate grids with `indices` and `meshgrid`

In [ ]:
matrix = np.arange(20).reshape(5, 4)

rows = np.array([0, 2, 4])
print(matrix[rows])

# Select one column index per row
column_indices = np.array([1, 2, 0, 3, 1])
print(np.take_along_axis(matrix, column_indices[:, None], axis=1))

# Coordinate grids
r, c = np.indices((3, 4))
print(r)
print(c)

## 17. Numerical computing concepts you should know

### Floating-point precision
Computers represent most real numbers approximately. Tiny rounding differences are normal. Compare floating-point arrays with `np.allclose` or `np.isclose`, not exact equality in most numerical applications.

### Stable calculations
- Prefer `np.log1p(x)` over `np.log(1 + x)` when `x` is very small.
- Prefer `np.expm1(x)` over `np.exp(x) - 1` when `x` is very small.
- Use `np.hypot(x, y)` for a stable Euclidean combination.
- Use `np.linalg.solve` rather than explicitly forming a matrix inverse.
- Be alert to overflow, underflow, division by zero and invalid operations.

In [ ]:
a = np.array([0.1 + 0.2])
b = np.array([0.3])
print(a == b)
print(np.isclose(a, b))
print(np.allclose(a, b))

small = 1e-12
print(np.log1p(small))
print(np.expm1(small))
print(np.hypot(3, 4))

## 18. NumPy for machine learning: core patterns

You should be comfortable with these before moving deeply into ML frameworks.

### Feature matrix convention
A dataset is commonly represented as `X` with shape `(n_samples, n_features)`, and targets as `y` with shape `(n_samples,)` or `(n_samples, n_outputs)`.

### Standardization
For each feature, subtract its training-set mean and divide by its training-set standard deviation. Compute these statistics on the training set only, then reuse them on validation/test data to avoid data leakage.

In [ ]:
rng = np.random.default_rng(12)
X = rng.normal(size=(100, 3)) * np.array([2., 10., 100.]) + np.array([5., 50., -20.])
y = 2 * X[:, 0] - 0.5 * X[:, 1] + rng.normal(size=100)

# Split first
order = rng.permutation(len(X))
train_idx, test_idx = order[:80], order[80:]
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

# Fit preprocessing statistics only on training data
mu = X_train.mean(axis=0)
sigma = X_train.std(axis=0)
sigma[sigma == 0] = 1

X_train_scaled = (X_train - mu) / sigma
X_test_scaled = (X_test - mu) / sigma

print(X_train_scaled.shape, X_test_scaled.shape)
print(X_train_scaled.mean(axis=0))
print(X_train_scaled.std(axis=0))

### Implement linear regression with NumPy

For linear regression with a bias term, one can solve the least-squares problem. This example uses `lstsq`, which is generally preferable to the normal-equation inverse formula.

In [ ]:
# Add a column of ones for the intercept.
X_design = np.column_stack([np.ones(len(X_train_scaled)), X_train_scaled])
weights, residuals, rank, singular_values = np.linalg.lstsq(X_design, y_train, rcond=None)

predictions = X_design @ weights
mse = np.mean((predictions - y_train) ** 2)

print("bias and weights:", weights)
print("training MSE:", mse)

### Logistic function, probabilities and classification metrics

In [ ]:
def sigmoid(z):
    # Clip to reduce overflow for this teaching example.
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

logits = np.array([-3., -1., 0., 1., 3.])
probabilities = sigmoid(logits)
predicted_classes = (probabilities >= 0.5).astype(int)
print(probabilities)
print(predicted_classes)

def accuracy(y_true, y_pred):
    return np.mean(np.asarray(y_true) == np.asarray(y_pred))

print(accuracy([0, 1, 1, 0], [0, 1, 0, 0]))

## 19. Useful NumPy utilities and inspection tools

- `np.info(obj)`: print information about an object.
- `np.lookfor("keyword")`: search NumPy's documentation strings.
- `np.show_config()`: show build and linked-library configuration.
- `np.set_printoptions(...)`: customize array display.
- `np.errstate(...)`: control floating-point warnings locally.
- `np.testing`: testing helpers such as `assert_array_equal` and `assert_allclose`.
- `np.save`, `np.load`: save/load `.npy` arrays.
- `np.savez`, `np.savez_compressed`: store multiple arrays in `.npz` archives.
- `np.savetxt`, `np.loadtxt`, `np.genfromtxt`: text-file I/O.

In [ ]:
np.set_printoptions(precision=3, suppress=True)
print(np.array([1/3, 1/7, 1/9]))

# Save/load demonstration using a temporary local filename.
sample = np.arange(6).reshape(2, 3)
np.save("sample_array.npy", sample)
loaded = np.load("sample_array.npy")
print(loaded)

np.testing.assert_array_equal(sample, loaded)

## 20. Function map: what to learn and when

This is a compact index of the API families introduced in this course. You do not need to memorize every name; learn to recognize the family and find the exact function quickly.

| Task | Functions / attributes |
|---|---|
| Create | `array`, `asarray`, `arange`, `linspace`, `zeros`, `ones`, `full`, `empty`, `eye`, `*_like` |
| Inspect | `shape`, `ndim`, `size`, `dtype`, `itemsize`, `nbytes` |
| Index | integer indexing, slices, Boolean masks, integer-array indexing |
| Reshape | `reshape`, `ravel`, `flatten`, `transpose`, `swapaxes`, `moveaxis`, `squeeze`, `expand_dims` |
| Combine | `concatenate`, `stack`, `vstack`, `hstack`, `column_stack`, `split` |
| Repeat/reorder | `repeat`, `tile`, `flip`, `roll`, `take`, `put` |
| Math | `add`, `subtract`, `multiply`, `divide`, `power`, `sqrt`, `exp`, `log`, trig ufuncs |
| Reduce | `sum`, `prod`, `min`, `max`, `mean`, `median`, `std`, `var`, `argmin`, `argmax`, `any`, `all` |
| Select | `where`, `select`, `unique`, `isin`, `nonzero`, `argwhere`, `clip`, `bincount` |
| Sort/search | `sort`, `argsort`, `partition`, `argpartition`, `searchsorted` |
| Random | `default_rng`, `random`, `integers`, `normal`, `choice`, `permutation`, `shuffle` |
| Linear algebra | `dot`, `matmul`, `outer`, `norm`, `solve`, `lstsq`, `svd`, `eig`, `det`, `inv`, `qr` |
| Statistics | `average`, `percentile`, `quantile`, `corrcoef`, `cov`, `histogram` |
| Missing data | `isnan`, `isfinite`, `nanmean`, `nansum`, `nan_to_num` |
| Files | `save`, `load`, `savez`, `savetxt`, `loadtxt`, `genfromtxt` |
| Testing | `allclose`, `isclose`, `np.testing.*` |

## 21. Practice lab — beginner to advanced

Try each task before opening the solution section. The goal is not just to get the output; explain the shape and dtype at each step.

1. Create an array containing the integers 0 through 99. Reshape it into 10 rows and 10 columns.
2. Extract the third row, fourth column, and the 3×3 center region.
3. Select all values divisible by 7.
4. Replace every negative value in an array with zero using `np.where`.
5. Normalize each column of a 2D array.
6. Generate a reproducible random 80/20 train/test index split.
7. Find the index of the largest value in each row.
8. Sort each row independently.
9. Count the frequency of integer labels 0 through 4.
10. Calculate the Euclidean length of each row vector.
11. Solve a system of linear equations using `np.linalg.solve`.
12. Fit a least-squares line to noisy data.
13. Implement mean squared error and accuracy.
14. Build a pairwise distance matrix between two sets of 2D points using broadcasting.
15. Explain why computing normalization statistics on the entire dataset before splitting can cause data leakage.

## 22. Practice lab — solutions

Read these only after attempting the exercises.

In [ ]:
# 1–2
grid = np.arange(100).reshape(10, 10)
third_row = grid[2]
fourth_column = grid[:, 3]
center = grid[3:6, 3:6]

# 3
multiples_of_7 = grid[grid % 7 == 0]

# 4
values = np.array([-3, 2, -1, 5])
nonnegative = np.where(values < 0, 0, values)

# 5
data = np.arange(15, dtype=float).reshape(5, 3)
column_std = data.std(axis=0, keepdims=True)
column_std[column_std == 0] = 1
normalized = (data - data.mean(axis=0, keepdims=True)) / column_std

# 6
rng = np.random.default_rng(123)
indices = rng.permutation(100)
train, test = indices[:80], indices[80:]

# 7–8
M = np.array([[2, 8, 1], [9, 3, 4]])
row_max_indices = np.argmax(M, axis=1)
sorted_rows = np.sort(M, axis=1)

# 9
labels = np.array([0, 1, 1, 2, 4, 2, 2, 4])
frequencies = np.bincount(labels, minlength=5)

# 10
vectors = np.array([[3., 4.], [5., 12.]])
lengths = np.linalg.norm(vectors, axis=1)

# 11
A = np.array([[2., 1.], [1., 3.]])
b = np.array([5., 6.])
solution = np.linalg.solve(A, b)

# 12
rng = np.random.default_rng(5)
x = np.linspace(0, 10, 100)
y = 3.5 * x + 2 + rng.normal(0, 1, size=x.shape)
design = np.column_stack([np.ones_like(x), x])
fit, *_ = np.linalg.lstsq(design, y, rcond=None)

# 13
def mean_squared_error(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.mean((y_true - y_pred) ** 2)

def classification_accuracy(y_true, y_pred):
    return np.mean(np.asarray(y_true) == np.asarray(y_pred))

# 14: pairwise Euclidean distances, output shape (n_A, n_B)
A_points = np.array([[0., 0.], [1., 1.]])
B_points = np.array([[2., 0.], [0., 3.], [1., 2.]])
differences = A_points[:, None, :] - B_points[None, :, :]
distances = np.linalg.norm(differences, axis=-1)

print("center region:\n", center)
print("multiples of 7:", multiples_of_7[:10])
print("frequencies:", frequencies)
print("vector lengths:", lengths)
print("linear fit [intercept, slope]:", fit)
print("pairwise distances:\n", distances)

## 23. Capstone projects

Complete these without copying a tutorial line by line.

### Project A — Data analysis from scratch
Generate or load a dataset with several numerical features. Calculate summary statistics, identify outliers, handle missing values, normalize features, and produce a short written report.

### Project B — Image arrays
An image is commonly represented as an array of shape `(height, width, channels)`. Load an image using an image library, inspect its dtype and range, crop it with slicing, flip it, and apply simple brightness or channel operations. Be careful about integer overflow and clipping.

### Project C — Linear regression from scratch
Implement prediction, mean squared error, and gradient descent using only NumPy. Track loss across iterations. Test on synthetic data where you know the true parameters.

### Project D — Mini neural-network components
Implement a dense layer forward pass, ReLU, sigmoid, softmax (with numerical stabilization), cross-entropy loss, and gradients. Verify gradients with finite differences on tiny arrays.

### Project E — Pairwise distances and nearest neighbours
Build a k-nearest-neighbours classifier using broadcasting or matrix operations. Compare your result with a trusted library after your own implementation works.

## 24. Mastery checklist

You are ready to move on when you can do these comfortably without searching for every line:

- [ ] Create arrays with the right shape and dtype.
- [ ] Predict the shape of indexing and slicing expressions.
- [ ] Explain views, copies and memory sharing.
- [ ] Reshape, transpose, concatenate and split arrays.
- [ ] Explain broadcasting from right to left.
- [ ] Use Boolean masks and integer-array indexing.
- [ ] Perform vectorized calculations and reductions along the correct axis.
- [ ] Handle NaN and infinity safely.
- [ ] Generate reproducible random data.
- [ ] Use matrix multiplication and solve linear systems.
- [ ] Standardize ML features without data leakage.
- [ ] Write and test numerical functions.
- [ ] Diagnose shape, dtype and numerical-stability bugs.
- [ ] Read the official documentation to learn unfamiliar functions.

## 25. Best official references

- NumPy documentation: https://numpy.org/doc/stable/
- NumPy quickstart: https://numpy.org/doc/stable/user/quickstart.html
- Absolute basics for beginners: https://numpy.org/doc/stable/user/absolute_beginners.html
- NumPy API reference: https://numpy.org/doc/stable/reference/
- NumPy linear algebra: https://numpy.org/doc/stable/reference/routines.linalg.html
- NumPy random sampling: https://numpy.org/doc/stable/reference/random/index.html

**A final learning rule:** don't aim to memorize the entire API. Aim to understand array shapes, axes, broadcasting, indexing, vectorization, memory behavior and numerical reasoning. Those ideas make the individual functions much easier to learn and transfer directly to pandas, scikit-learn, PyTorch and scientific computing.